# Enhanced Baseline Experiments Analysis

## Comprehensive Analysis of Intrinsic Dimension Estimates

This notebook provides thorough analysis of baseline intervention experiments from `exp3_section2_baselines.py`. It examines how different interventions affect the intrinsic dimensionality of transformer latent representations across layers.

### Key Analyses Provided:
- **Depth-wise ID trajectories** per baseline intervention with confidence intervals
- **Statistical comparisons** across baselines and estimators with comprehensive metrics
- **Intervention effect analysis** with effect sizes relative to reference baselines
- **Robust error handling** for missing or incomplete data
- **Comprehensive visualizations** with proper error bars and sample counts
- **Model comparison** across different architectures
- **Data validation and cleaning** with outlier detection

### Data Sources:
- `results/baselines/` directory containing `id.jsonl` files from baseline experiments
- Automatic discovery and loading of all experiment runs
- Support for multiple models, baselines, and estimators
- Robust metadata extraction from file paths and manifest files

### Baseline Interventions Analyzed:
- **syntactic_disintegration**: Token permutation preserving sentence structure
- **semantic_scrambling**: Random token replacement
- **maximum_entropy_injection**: Gaussian noise input (trained models only)
- **topological_initialisation**: LayerNorm affine reset (trained models only)

### Estimators:
- **TwoNN**: Fast, parameter-free estimator
- **ESS**: Local structure-aware estimator
- **Participation Ratio**: Participation ratio estimator

### Key Improvements:
- **Robust data loading** with fallback metadata extraction
- **Data validation** with outlier detection and error analysis
- **Statistical testing** between baselines with significance tests
- **Intervention effects** quantified as percentage changes
- **Export capabilities** for cleaned data and summary reports
- **Professional visualizations** with proper error bars and legends

In [ ]:
import json
import warnings
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from IPython.display import display, HTML

# Configure plotting
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.style.use('seaborn-v0_8')

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("Enhanced Baseline Analysis Notebook")
print("====================================")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Seaborn available: {'Yes' if 'sns' in globals() else 'No'}")
print(f"SciPy available: {'Yes' if 'stats' in dir() else 'No'}")
print()

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
RESULTS_ROOT = Path("results/baselines")
ESTIMATORS = ["twonn", "ess", "participation_ratio"]
CONFIDENCE_LEVEL = 0.95  # For confidence intervals
MIN_SAMPLES_FOR_STATS = 3  # Minimum samples for statistical analysis

# Color scheme for consistent plotting
BASELINE_COLORS = {
    'syntactic_disintegration': '#1f77b4',  # Blue
    'semantic_scrambling': '#ff7f0e',       # Orange
    'maximum_entropy_injection': '#2ca02c', # Green
    'topological_initialisation': '#d62728' # Red
}

ESTIMATOR_COLORS = {
    'twonn': '#1f77b4',
    'ess': '#ff7f0e',
    'participation_ratio': '#2ca02c'
}

print(f"Configuration:")
print(f"  Results root: {RESULTS_ROOT}")
print(f"  Estimators: {ESTIMATORS}")
print(f"  Confidence level: {CONFIDENCE_LEVEL}")
print(f"  Min samples for stats: {MIN_SAMPLES_FOR_STATS}")
print(f"  Available baseline colors: {list(BASELINE_COLORS.keys())}")

In [ ]:
# ── Enhanced Data Loading ──────────────────────────────────────────────
def load_baseline_results(results_root: Path):
    """Load all id.jsonl files from baseline experiment runs with robust metadata extraction."""
    all_rows = []
    run_info = []
    
    # Find all id.jsonl files
    jsonl_files = list(results_root.rglob("id.jsonl"))
    print(f"Found {len(jsonl_files)} id.jsonl files to process")
    
    for jsonl_path in jsonl_files:
        try:
            # Check if file has content
            if jsonl_path.stat().st_size == 0:
                print(f"  Skipping empty file: {jsonl_path.relative_to(results_root)}")
                continue
                
            with open(jsonl_path, 'r') as f:
                rows = [json.loads(line) for line in f if line.strip()]
            
            if not rows:
                print(f"  Skipping file with no valid rows: {jsonl_path.relative_to(results_root)}")
                continue
            
            # Extract metadata from path and manifest
            rel_path = jsonl_path.relative_to(results_root)
            parts = rel_path.parts
            
            # Default metadata
            metadata = {
                'run_group': 'unknown',
                'model': 'unknown',
                'run_id': 'unknown',
                'weights': 'unknown',
                'n_layers': 'unknown',
                'd_model': 'unknown'
            }
            
            # Try to parse from path
            if len(parts) >= 3:
                metadata.update({
                    'run_group': parts[0],
                    'model': parts[1],
                    'run_id': parts[2]
                })
            
            # Try to get additional metadata from manifest
            manifest_path = jsonl_path.parent / "manifest.json"
            if manifest_path.exists():
                try:
                    with open(manifest_path, 'r') as f:
                        manifest = json.load(f)
                        config = manifest.get('config', {})
                        metadata.update({
                            'weights': config.get('weights', metadata['weights']),
                            'n_layers': config.get('n_layers', metadata['n_layers']),
                            'd_model': config.get('d_model', metadata['d_model'])
                        })
                except Exception as e:
                    print(f"  Warning: Could not read manifest {manifest_path}: {e}")
            else:
                print(f"  Warning: No manifest found at {manifest_path}")
            
            # Add metadata to all rows
            for row in rows:
                row.update(metadata)
            
            all_rows.extend(rows)
            run_info.append({
                'path': str(rel_path),
                'n_rows': len(rows),
                'metadata': metadata
            })
            
            print(f"  ✓ Loaded {len(rows)} rows from {rel_path}")
            print(f"    Model: {metadata['model']}, Weights: {metadata['weights']}")
            
        except Exception as e:
            print(f"  ✗ Error loading {jsonl_path.relative_to(results_root)}: {e}")
    
    df = pd.DataFrame(all_rows)
    
    # Summary of loaded runs
    print(f"\nLoaded {len(run_info)} successful runs:")
    for info in run_info:
        meta = info['metadata']
        print(f"  - {info['path']}: {info['n_rows']} rows ({meta['model']}, {meta['weights']})")
    
    return df

# Load data with error handling
print("Loading baseline experiment data...")
try:
    df = load_baseline_results(RESULTS_ROOT)
    print(f"\n✅ Data loading complete!")
    print(f"Total rows loaded: {len(df)}")
    
    # Safe column access with fallbacks
    if len(df) > 0:
        if 'model' in df.columns:
            print(f"Unique models: {sorted(df['model'].unique())}")
        else:
            print("⚠️  Warning: 'model' column not found")
            
        if 'baseline' in df.columns:
            print(f"Unique baselines: {sorted(df['baseline'].unique())}")
        else:
            print("⚠️  Warning: 'baseline' column not found")
            
        if 'depth' in df.columns:
            print(f"Depth range: {df['depth'].min()} to {df['depth'].max()}")
        else:
            print("⚠️  Warning: 'depth' column not found")
            
        if 'granularity' in df.columns:
            print(f"Granularities: {df['granularity'].unique()}")
        else:
            print("⚠️  Warning: 'granularity' column not found")
            
        if 'estimator' in df.columns:
            print(f"Estimators: {sorted(df['estimator'].unique())}")
        else:
            print("⚠️  Warning: 'estimator' column not found")
    else:
        print("❌ No data loaded! Check results directory and file formats.")
        
except Exception as e:
    print(f"❌ Critical error during data loading: {e}")
    df = pd.DataFrame()  # Empty dataframe as fallback

In [ ]:
# ── Data Validation and Cleaning ──────────────────────────────────────
def validate_and_clean_data(df):
    """Validate data quality and clean common issues."""
    if df.empty:
        print("No data to validate")
        return df
    
    print("Data Validation:")
    print("-" * 20)
    
    # Check for required columns
    required_cols = ['baseline', 'depth', 'estimator', 'id_estimate']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"❌ Missing required columns: {missing_cols}")
        return df[[]]  # Return empty dataframe
    
    # Check data types and ranges
    print(f"Data shape: {df.shape}")
    print(f"Data types:")
    for col in required_cols:
        dtype = df[col].dtype
        print(f"  {col}: {dtype}")
    
    # Clean data
    df_clean = df.copy()
    
    # Handle ID estimates
    if 'id_estimate' in df_clean.columns:
        # Remove NaN and infinite values
        original_count = len(df_clean)
        df_clean = df_clean.dropna(subset=['id_estimate'])
        df_clean = df_clean[np.isfinite(df_clean['id_estimate'])]
        
        if len(df_clean) < original_count:
            removed = original_count - len(df_clean)
            print(f"Removed {removed} rows with invalid ID estimates")
        
        # Check for reasonable ID ranges
        id_min, id_max = df_clean['id_estimate'].min(), df_clean['id_estimate'].max()
        print(f"ID estimate range: {id_min:.2f} - {id_max:.2f}")
        
        # Flag potential outliers
        if id_max > 1000:
            high_count = (df_clean['id_estimate'] > 1000).sum()
            print(f"⚠️  {high_count} estimates > 1000 (potential outliers)")
    
    # Check for errors
    if 'error' in df_clean.columns:
        error_count = df_clean['error'].notna().sum()
        if error_count > 0:
            print(f"⚠️  {error_count} rows have error messages")
            error_types = df_clean['error'].dropna().value_counts()
            print(f"Error types: {dict(error_types)}")
    
    print(f"\n✅ Validation complete. {len(df_clean)} valid rows remaining.")
    return df_clean

# Validate and clean data
df_clean = validate_and_clean_data(df)

# Show basic statistics
if not df_clean.empty:
    print("\nBasic Statistics:")
    print("=================")
    display(df_clean.describe())
    
    print("\nData Completeness:")
    print("===================")
    completeness = (1 - df_clean.isnull().sum() / len(df_clean)) * 100
    print(completeness.round(1).astype(str) + '%')

In [ ]:
# ── Depth Trajectory Analysis ─────────────────────────────────────────
def plot_depth_trajectories(df, estimator='twonn', baseline_filter=None, save_path=None):
    """Plot ID estimates vs depth for each baseline with confidence intervals."""
    
    if df.empty:
        print("No data to plot")
        return
    
    # Filter data
    plot_df = df[df['estimator'] == estimator].copy()
    if baseline_filter:
        plot_df = plot_df[plot_df['baseline'].isin(baseline_filter)]
    
    if plot_df.empty:
        print(f"No data for estimator '{estimator}'")
        return
    
    # Group by baseline and depth for statistics
    baselines = sorted(plot_df['baseline'].unique())
    n_baselines = len(baselines)
    
    # Create subplots
    if n_baselines <= 2:
        fig, axes = plt.subplots(1, n_baselines, figsize=(6*n_baselines, 5))
        if n_baselines == 1:
            axes = [axes]
    else:
        fig, axes = plt.subplots(2, (n_baselines+1)//2, figsize=(12, 8))
        axes = axes.flatten()
    
    for i, baseline in enumerate(baselines):
        ax = axes[i] if n_baselines > 1 else axes[0]
        
        baseline_data = plot_df[plot_df['baseline'] == baseline]
        
        # Group by depth and compute statistics
        depth_stats = baseline_data.groupby('depth')['id_estimate'].agg(
            ['mean', 'std', 'count', 'min', 'max']
        ).reset_index()
        
        # Filter depths with sufficient samples
        depth_stats = depth_stats[depth_stats['count'] >= MIN_SAMPLES_FOR_STATS]
        
        if depth_stats.empty:
            ax.text(0.5, 0.5, 'Insufficient data', 
                   ha='center', va='center', transform=ax.transAxes)
            continue
        
        # Plot with confidence intervals
        x = depth_stats['depth']
        y = depth_stats['mean']
        yerr = depth_stats['std']
        
        color = BASELINE_COLORS.get(baseline, '#333333')
        
        ax.errorbar(x, y, yerr=yerr, 
                   marker='o', markersize=6, linewidth=2, capsize=4,
                   color=color, ecolor=color, alpha=0.8,
                   label=f"{baseline.replace('_', ' ')} (n={len(baseline_data)})")
        
        # Add individual points as scatter
        for _, row in depth_stats.iterrows():
            depth_data = baseline_data[baseline_data['depth'] == row['depth']]
            ax.scatter([row['depth']] * len(depth_data), depth_data['id_estimate'], 
                      color=color, alpha=0.3, s=20)
        
        # Formatting
        baseline_name = baseline.replace('_', ' ').title()
        ax.set_title(f'{baseline_name}\n(n={len(baseline_data)} points)', fontsize=12)
        ax.set_xlabel('Transformer Depth', fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(x.min()-0.5, x.max()+0.5)
    
    # Set common ylabel
    if n_baselines == 1:
        axes[0].set_ylabel(f'Intrinsic Dimension ({estimator.upper()})', fontsize=11)
    else:
        fig.text(0.04, 0.5, f'Intrinsic Dimension ({estimator.upper()})', 
                va='center', rotation='vertical', fontsize=11)
    
    plt.suptitle(f'Depth Trajectories: {estimator.upper()} Estimator', fontsize=14, y=0.95)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved plot: {save_path}")
    
    plt.show()

# Plot depth trajectories for each estimator
if not df_clean.empty:
    for estimator in ESTIMATORS:
        if estimator in df_clean['estimator'].values:
            print(f"\n=== {estimator.upper()} Depth Trajectories ===")
            plot_depth_trajectories(df_clean, estimator=estimator)

In [ ]:
# ── Baseline Comparison Analysis ──────────────────────────────────────
def plot_baseline_comparison(df, depth=0, estimator='twonn', save_path=None):
    """Compare ID estimates across baselines at a specific depth."""
    
    if df.empty:
        print("No data to plot")
        return
    
    # Filter data
    plot_df = df[
        (df['depth'] == depth) & 
        (df['estimator'] == estimator) &
        (df['granularity'] == 'last_token')
    ].dropna(subset=['id_estimate'])
    
    if plot_df.empty:
        print(f"No data for depth {depth}, estimator {estimator}")
        return
    
    # Group by model and baseline for comparison
    models = sorted(plot_df['model'].unique())
    baselines = sorted(plot_df['baseline'].unique())
    
    fig, axes = plt.subplots(1, len(models), figsize=(5*len(models), 6), sharey=True)
    if len(models) == 1:
        axes = [axes]
    
    for i, model in enumerate(models):
        ax = axes[i]
        model_data = plot_df[plot_df['model'] == model]
        
        # Prepare data for bar plot
        baseline_means = []
        baseline_errors = []
        baseline_counts = []
        
        for baseline in baselines:
            baseline_data = model_data[model_data['baseline'] == baseline]
            if len(baseline_data) >= MIN_SAMPLES_FOR_STATS:
                baseline_means.append(baseline_data['id_estimate'].mean())
                baseline_errors.append(baseline_data['id_estimate'].std())
                baseline_counts.append(len(baseline_data))
            else:
                baseline_means.append(np.nan)
                baseline_errors.append(0)
                baseline_counts.append(0)
        
        # Filter out NaN values
        valid_indices = [i for i, mean in enumerate(baseline_means) if not np.isnan(mean)]
        valid_baselines = [baselines[i] for i in valid_indices]
        valid_means = [baseline_means[i] for i in valid_indices]
        valid_errors = [baseline_errors[i] for i in valid_indices]
        
        if valid_means:
            bars = ax.bar(range(len(valid_means)), valid_means, 
                         yerr=valid_errors, capsize=5, alpha=0.8)
            
            # Color bars by baseline
            for j, bar in enumerate(bars):
                baseline_name = valid_baselines[j]
                bar.set_color(BASELINE_COLORS.get(baseline_name, '#333333'))
                
                # Add sample count annotation
                count = baseline_counts[valid_indices[j]]
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2, 
                       height + (max(valid_errors) if valid_errors else 0) + max(valid_means)*0.02,
                       f'n={count}', ha='center', va='bottom', fontsize=9)
        
        # Formatting
        ax.set_title(f'{model}\nDepth {depth}', fontsize=12)
        ax.set_xticks(range(len(valid_baselines)))
        ax.set_xticklabels([b.replace('_', '\n') for b in valid_baselines], 
                          rotation=45, ha='right', fontsize=10)
        ax.grid(True, axis='y', alpha=0.3)
    
    # Set common ylabel
    fig.text(0.04, 0.5, f'Intrinsic Dimension\n({estimator.upper()})', 
             va='center', rotation='vertical', fontsize=11)
    
    plt.suptitle(f'Baseline Comparison at Depth {depth}', fontsize=14, y=0.98)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved comparison plot: {save_path}")
    
    plt.show()

# Plot baseline comparisons
if not df_clean.empty:
    for estimator in ESTIMATORS:
        if estimator in df_clean['estimator'].values:
            print(f"\n=== {estimator.upper()} Baseline Comparison ===")
            plot_baseline_comparison(df_clean, depth=0, estimator=estimator)

In [ ]:
# ── Comprehensive Statistical Analysis ────────────────────────────────
def comprehensive_stats(df):
    """Provide comprehensive statistical analysis of the ID estimates."""
    
    if df.empty:
        print("No data for statistical analysis")
        return
    
    print("Comprehensive Statistical Analysis")
    print("=" * 40)
    
    # Overall statistics
    valid_data = df.dropna(subset=['id_estimate'])
    print(f"\n📊 Overall Statistics:")
    print(f"   Total measurements: {len(valid_data)}")
    print(f"   ID range: {valid_data['id_estimate'].min():.2f} - {valid_data['id_estimate'].max():.2f}")
    print(f"   Mean ± std: {valid_data['id_estimate'].mean():.2f} ± {valid_data['id_estimate'].std():.2f}")
    
    # Per-estimator analysis
    print(f"\n🔍 Per-Estimator Analysis:")
    estimator_stats = valid_data.groupby('estimator')['id_estimate'].agg(
        ['count', 'mean', 'std', 'min', 'max', 'median']
    )
    print(estimator_stats.round(2))
    
    # Per-baseline analysis
    print(f"\n🎯 Per-Baseline Analysis:")
    baseline_stats = valid_data.groupby('baseline')['id_estimate'].agg(
        ['count', 'mean', 'std', 'min', 'max', 'median']
    )
    print(baseline_stats.round(2))
    
    # Depth-wise analysis
    print(f"\n📈 Depth-wise Analysis:")
    depth_stats = valid_data.groupby('depth')['id_estimate'].agg(
        ['count', 'mean', 'std']
    )
    print(depth_stats.round(2))
    
    # Model comparison (if multiple models)
    if 'model' in valid_data.columns and len(valid_data['model'].unique()) > 1:
        print(f"\n🤖 Model Comparison:")
        model_stats = valid_data.groupby('model')['id_estimate'].agg(
            ['count', 'mean', 'std']
        )
        print(model_stats.round(2))
    
    # Statistical tests between baselines
    print(f"\n⚗️  Statistical Tests:")
    baselines = valid_data['baseline'].unique()
    if len(baselines) >= 2:
        for i, baseline1 in enumerate(baselines):
            for baseline2 in baselines[i+1:]:
                data1 = valid_data[valid_data['baseline'] == baseline1]['id_estimate']
                data2 = valid_data[valid_data['baseline'] == baseline2]['id_estimate']
                
                if len(data1) >= MIN_SAMPLES_FOR_STATS and len(data2) >= MIN_SAMPLES_FOR_STATS:
                    # t-test
                    t_stat, p_val = stats.ttest_ind(data1, data2, equal_var=False)
                    print(f"   {baseline1[:15]:<15} vs {baseline2[:15]:<15}: t={t_stat:.2f}, p={p_val:.3f}")
    
    # Error analysis
    if 'error' in df.columns:
        error_rows = df['error'].notna()
        error_count = error_rows.sum()
        if error_count > 0:
            print(f"\n❌ Error Analysis:")
            print(f"   Total errors: {error_count} ({error_count/len(df)*100:.1f}%)")
            error_types = df[error_rows]['error'].value_counts()
            print(f"   Error types:")
            for error_type, count in error_types.items():
                print(f"     - {error_type}: {count}")

if not df_clean.empty:
    comprehensive_stats(df_clean)

In [ ]:
# ── Intervention Effect Analysis ─────────────────────────────────────
def analyze_intervention_effects(df, reference_baseline='trained', estimator='twonn'):
    """Analyze the effect of interventions relative to a reference baseline."""
    
    if df.empty or 'model' not in df.columns:
        print("Insufficient data for intervention effect analysis")
        return
    
    print(f"Intervention Effect Analysis (Reference: {reference_baseline})")
    print("=" * 60)
    
    # Filter to specific estimator
    effect_df = df[df['estimator'] == estimator].copy()
    
    # Group by model for analysis
    models = effect_df['model'].unique()
    
    for model in models:
        model_data = effect_df[effect_df['model'] == model]
        
        print(f"\n📊 Model: {model}")
        print("-" * 30)
        
        # Check if reference baseline exists
        if reference_baseline not in model_data['baseline'].values:
            print(f"   Reference baseline '{reference_baseline}' not found for this model")
            continue
        
        # Get reference values (mean per depth)
        ref_data = model_data[model_data['baseline'] == reference_baseline]
        ref_means = ref_data.groupby('depth')['id_estimate'].mean()
        
        # Analyze each intervention
        interventions = [b for b in model_data['baseline'].unique() if b != reference_baseline]
        
        for intervention in interventions:
            int_data = model_data[model_data['baseline'] == intervention]
            int_means = int_data.groupby('depth')['id_estimate'].mean()
            
            # Calculate effect sizes
            common_depths = set(ref_means.index) & set(int_means.index)
            if not common_depths:
                continue
            
            effects = []
            for depth in common_depths:
                ref_val = ref_means[depth]
                int_val = int_means[depth]
                effect = (int_val - ref_val) / ref_val * 100  # Percentage change
                effects.append(effect)
            
            if effects:
                mean_effect = np.mean(effects)
                std_effect = np.std(effects)
                print(f"   {intervention.replace('_', ' '):<20}: {mean_effect:+.1f}% ± {std_effect:.1f}%")
    
    # Visualization
    if len(models) > 0:
        plot_intervention_effects(effect_df, reference_baseline, estimator)

def plot_intervention_effects(df, reference_baseline, estimator):
    """Visualize intervention effects relative to reference."""
    
    # Calculate effect sizes per model and baseline
    effect_data = []
    
    models = df['model'].unique()
    for model in models:
        model_data = df[df['model'] == model]
        
        if reference_baseline not in model_data['baseline'].values:
            continue
        
        ref_data = model_data[model_data['baseline'] == reference_baseline]
        ref_means = ref_data.groupby('depth')['id_estimate'].mean()
        
        interventions = [b for b in model_data['baseline'].unique() if b != reference_baseline]
        for intervention in interventions:
            int_data = model_data[model_data['baseline'] == intervention]
            int_means = int_data.groupby('depth')['id_estimate'].mean()
            
            common_depths = set(ref_means.index) & set(int_means.index)
            for depth in common_depths:
                ref_val = ref_means[depth]
                int_val = int_means[depth]
                effect = (int_val - ref_val) / ref_val * 100
                effect_data.append({
                    'model': model,
                    'intervention': intervention,
                    'depth': depth,
                    'effect_percent': effect
                })
    
    if not effect_data:
        return
    
    effect_df = pd.DataFrame(effect_data)
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Group by intervention and plot
    interventions = effect_df['intervention'].unique()
    for intervention in interventions:
        int_data = effect_df[effect_df['intervention'] == intervention]
        ax.scatter(int_data['depth'], int_data['effect_percent'], 
                  label=intervention.replace('_', ' '), alpha=0.7, s=50)
        
        # Add trend line if multiple points
        if len(int_data) > 2:
            z = np.polyfit(int_data['depth'], int_data['effect_percent'], 1)
            p = np.poly1d(z)
            x_trend = np.linspace(int_data['depth'].min(), int_data['depth'].max(), 50)
            ax.plot(x_trend, p(x_trend), alpha=0.5)
    
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.5, label='No effect')
    ax.set_xlabel('Transformer Depth')
    ax.set_ylabel(f'ID Change vs {reference_baseline} (%)')
    ax.set_title(f'Intervention Effects Relative to {reference_baseline}\n({estimator.upper()} Estimator)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Analyze intervention effects (if reference baseline exists)
if not df_clean.empty and 'trained' in df_clean['baseline'].values:
    print("\n" + "="*60)
    analyze_intervention_effects(df_clean, reference_baseline='trained', estimator='twonn')
else:
    print("\n⚠️  Skipping intervention effect analysis (no 'trained' baseline found)")

In [ ]:
# ── Summary and Export ───────────────────────────────────────────────
def create_summary_report(df):
    """Create a comprehensive summary report of the analysis."""
    
    if df.empty:
        return "No data available for summary report."
    
    report = []
    report.append("# Baseline Experiment Analysis Summary")
    report.append("=" * 50)
    report.append("")
    
    # Basic statistics
    report.append("## Overview")
    report.append(f"- Total measurements: {len(df)}")
    report.append(f"- Models analyzed: {', '.join(sorted(df['model'].unique()))}")
    report.append(f"- Baselines tested: {', '.join(sorted(df['baseline'].unique()))}")
    report.append(f"- Estimators used: {', '.join(sorted(df['estimator'].unique()))}")
    report.append(f"- Depth range: {df['depth'].min()} - {df['depth'].max()}")
    report.append("")
    
    # Key findings
    report.append("## Key Findings")
    
    # ID estimate ranges
    id_stats = df.groupby('estimator')['id_estimate'].agg(['min', 'max', 'mean', 'std'])
    for estimator, stats in id_stats.iterrows():
        report.append(f"- **{estimator.upper()}**: {stats['mean']:.1f} ± {stats['std']:.1f} "
                     f"(range: {stats['min']:.1f} - {stats['max']:.1f})")
    
    # Baseline differences
    baseline_stats = df.groupby('baseline')['id_estimate'].agg(['mean', 'count'])
    report.append("\n- **Baseline comparison**:")
    for baseline, stats in baseline_stats.iterrows():
        report.append(f"  - {baseline.replace('_', ' ')}: {stats['mean']:.1f} "
                     f"(n={int(stats['count'])})")
    
    report.append("")
    
    # Depth trends
    depth_trend = df.groupby('depth')['id_estimate'].mean()
    if len(depth_trend) > 1:
        trend_direction = "increasing" if depth_trend.iloc[-1] > depth_trend.iloc[0] else "decreasing"
        report.append(f"- **Depth trend**: ID estimates {trend_direction} with transformer depth")
    
    report.append("")
    
    # Recommendations
    report.append("## Recommendations")
    report.append("- Focus analysis on " + 
                 baseline_stats['mean'].idxmax().replace('_', ' ') + 
                 " for highest ID estimates")
    report.append("- Consider " + 
                 id_stats['std'].idxmin() + 
                 " for most consistent measurements")
    
    if 'error' in df.columns and df['error'].notna().any():
        error_rate = df['error'].notna().sum() / len(df) * 100
        report.append(f"- Address {error_rate:.1f}% estimation errors for improved reliability")
    
    return "\n".join(report)

# Generate and display summary
if not df_clean.empty:
    summary_report = create_summary_report(df_clean)
    print("\n" + "="*60)
    print(summary_report)
    
    # Export to file
    summary_path = RESULTS_ROOT / "analysis_summary.md"
    with open(summary_path, 'w') as f:
        f.write(summary_report)
    print(f"\n📄 Summary exported to: {summary_path}")
    
    # Export cleaned data
    data_export_path = RESULTS_ROOT / "cleaned_data.csv"
    df_clean.to_csv(data_export_path, index=False)
    print(f"📊 Cleaned data exported to: {data_export_path}")

print("\n🎉 Analysis Complete!")
print("Check the plots above and exported files for detailed results.")
print("\nNext steps:")
print("1. Review the depth trajectory plots for ID evolution patterns")
print("2. Examine baseline comparisons to understand intervention effects")
print("3. Check statistical significance of differences between baselines")
print("4. Consider running additional experiments with different parameters")